# Retail Sales Forecasting & Business Intelligence
## Notebook 02: Exploratory Data Analysis (EDA)

**Objective:** Uncover multi-frequency demand patterns, seasonal cycles, promotional price elasticity, return drivers, and SKU profitability across physical stores and categories.

**Data Assets Used:**
- `data/processed/retail_sales_clean.csv`: Full enterprise transaction log including customer returns (`SALE` & `RETURN`) for P&L, net revenue, and margin analysis.
- `data/processed/retail_sales_demand.csv`: Demand-only transactions (`SALE` only) for clean unpolluted time-series seasonality modeling.

**Metric Clarification:**
- Because transactions are recorded at daily store-product line grain without an `order_id` session identifier, basket size is evaluated as **Average Transaction Value (ATV = Net Revenue / Total Sale Rows)** rather than Average Order Value (AOV).

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add root and src to path
sys.path.append(os.path.abspath(".."))
from src.eda_visualizer import (
    plot_monthly_revenue_and_profit,
    plot_daily_moving_averages,
    plot_seasonality_grid,
    plot_category_performance_matrix,
    plot_pareto_revenue_analysis,
    plot_top_bottom_products,
    plot_store_and_regional_performance,
    plot_promotion_elasticity_and_lift,
    plot_returns_deep_dive,
    plot_correlation_and_distributions,
    generate_all_eda_reports
)

# Load clean datasets
df_clean = pd.read_csv("../data/processed/retail_sales_clean.csv", parse_dates=["order_date"])
df_demand = pd.read_csv("../data/processed/retail_sales_demand.csv", parse_dates=["order_date"])

print(f"Loaded clean dataset: {df_clean.shape[0]:,} rows | Demand dataset: {df_demand.shape[0]:,} rows")

### A. Executive KPI Summary Table
Computes dynamic financial KPIs directly from the ingested dataset and exports the results to `reports/eda_executive_summary.csv`.

In [2]:
# Robust split-apply-combine aggregation
gross_sales = float(df_clean.loc[df_clean["transaction_type"] == "SALE", "revenue"].sum())
returns_val = float(df_clean.loc[df_clean["transaction_type"] == "RETURN", "revenue"].abs().sum())
net_revenue = float(df_clean["revenue"].sum())
gross_profit = float(df_clean.loc[df_clean["transaction_type"] == "SALE", "profit"].sum())
net_profit = float(df_clean["profit"].sum())
units_sold = int(df_clean.loc[df_clean["transaction_type"] == "SALE", "units_sold"].sum())
sale_tx_count = int((df_clean["transaction_type"] == "SALE").sum())
return_tx_count = int((df_clean["transaction_type"] == "RETURN").sum())

return_rate = (returns_val / gross_sales * 100.0) if gross_sales > 0 else 0.0
profit_margin = (net_profit / net_revenue * 100.0) if net_revenue > 0 else 0.0
atv = (net_revenue / sale_tx_count) if sale_tx_count > 0 else 0.0

kpi_df = pd.DataFrame({
    "Metric": [
        "Total Gross Sales Revenue ($)",
        "Customer Return Refunds ($)",
        "Total Net Revenue ($)",
        "Gross Profit ($)",
        "Net Profit ($)",
        "Forward Units Sold",
        "Customer Return Rate (% of Gross)",
        "Blended Net Profit Margin (%)",
        "Average Transaction Value (ATV) ($)",
        "Sale Transaction Rows",
        "Return Transaction Rows"
    ],
    "Value": [
        f"${gross_sales:,.2f}",
        f"${returns_val:,.2f}",
        f"${net_revenue:,.2f}",
        f"${gross_profit:,.2f}",
        f"${net_profit:,.2f}",
        f"{units_sold:,}",
        f"{return_rate:.2f}%",
        f"{profit_margin:.2f}%",
        f"${atv:.2f}",
        f"{sale_tx_count:,}",
        f"{return_tx_count:,}"
    ]
})
display(kpi_df)

# Export dynamic summary CSV
os.makedirs("../reports", exist_ok=True)
kpi_df.to_csv("../reports/eda_executive_summary.csv", index=False)
print("Saved executive summary table to ../reports/eda_executive_summary.csv")

### B. Time Trends & Seasonality Decomposition
Uses vertically stacked subplots to compare monthly financial volumes against customer return rates without deceptive dual-axis scaling.

In [3]:
plot_monthly_revenue_and_profit(df_clean, output_dir="../reports/figures")
plot_daily_moving_averages(df_demand, output_dir="../reports/figures")
plot_seasonality_grid(df_clean, output_dir="../reports/figures")
plt.show()

### C. Merchandising Category & Product Pareto (80/20)
Evaluates category profitability, Pareto cumulative curve, and Top vs Bottom products via grouped horizontal bars.

In [4]:
plot_category_performance_matrix(df_clean, output_dir="../reports/figures")
plot_pareto_revenue_analysis(df_clean, output_dir="../reports/figures")
plot_top_bottom_products(df_clean, output_dir="../reports/figures")
plt.show()

### D. Store Network & Regional Performance
Examines store performance across regions and cross-tabulates category sales in a heatmap.

In [5]:
plot_store_and_regional_performance(df_clean, output_dir="../reports/figures")
plt.show()

### E. Promotional Price Elasticity & Discount Tiers
Analyzes order lift across discount bands: 0% Full Price, 10% Discount, 15% Discount, 20% Discount, and 30% Clearance.

In [6]:
plot_promotion_elasticity_and_lift(df_clean, output_dir="../reports/figures")
plt.show()

### F. Customer Return Rates & Financial Dilution
Quantifies return rates by category and assesses the behavioral link between promotions and return frequency.

In [7]:
plot_returns_deep_dive(df_clean, output_dir="../reports/figures")
plt.show()

### G. Behavioral Feature Correlations & Revenue Distribution
Focuses strictly on non-trivial behavioral features (`units_sold`, `unit_price`, `discount`, `promotion_flag`, `holiday_flag`, `is_weekend`) to avoid trivial accounting identity collinearity.

In [8]:
plot_correlation_and_distributions(df_demand, output_dir="../reports/figures")
plt.show()